# Naive Bayes spam classifier

A Bernoulli naive Bayes classifier written from scratch with NumPy, then compared against standard baselines.

Each email is described by 54 binary features: whether a keyword (e.g. *money*, *free*, *receive*) or a character (e.g. `!`, `$`) appears in it. The task is to predict spam (`1`) or ham (`0`).

In [1]:
import numpy as np

training_spam = np.loadtxt("data/training_spam.csv", delimiter=",").astype(int)
testing_spam = np.loadtxt("data/testing_spam.csv", delimiter=",").astype(int)

# Column 0 is the label (1 = spam, 0 = ham); columns 1-54 are the binary features.
X_train, y_train = training_spam[:, 1:], training_spam[:, 0]
X_test, y_test = testing_spam[:, 1:], testing_spam[:, 0]

print(f"Training set: {X_train.shape[0]} emails, {X_train.shape[1]} features, {y_train.mean():.1%} spam")
print(f"Test set:     {X_test.shape[0]} emails, {y_test.mean():.1%} spam")

Training set: 1000 emails, 54 features, 38.7% spam
Test set:     500 emails, 39.8% spam


## The model

For each class $c$ the classifier estimates the prior $P(c)$ and, for every feature $i$, the probability that the feature is present, $P(x_i = 1 \mid c)$. Laplace smoothing (add one to the counts) stops a feature that never appears in one class from giving a probability of zero.

To classify an email it compares

$$\log P(c) + \sum_i \big[x_i \log P(x_i{=}1 \mid c) + (1 - x_i) \log P(x_i{=}0 \mid c)\big]$$

for spam and ham. Working in log space avoids multiplying 54 small probabilities together, which would underflow.

In [2]:
class SpamClassifier:
    def train(self, X, y):
        spam, ham = X[y == 1], X[y == 0]
        self.log_prior_spam = np.log(len(spam) / len(X))
        self.log_prior_ham = np.log(len(ham) / len(X))
        # Laplace-smoothed probability that each feature is present, per class
        self.p_spam = (spam.sum(axis=0) + 1) / (len(spam) + 2)
        self.p_ham = (ham.sum(axis=0) + 1) / (len(ham) + 2)
        return self

    def log_likelihoods(self, X):
        log_spam = self.log_prior_spam + X @ np.log(self.p_spam) + (1 - X) @ np.log(1 - self.p_spam)
        log_ham = self.log_prior_ham + X @ np.log(self.p_ham) + (1 - X) @ np.log(1 - self.p_ham)
        return log_spam, log_ham

    def predict(self, X):
        log_spam, log_ham = self.log_likelihoods(X)
        return (log_spam > log_ham).astype(int)


classifier = SpamClassifier().train(X_train, y_train)

## Results on the held-out test set

In [3]:
predictions = classifier.predict(X_test)

tp = np.sum((predictions == 1) & (y_test == 1))
fp = np.sum((predictions == 1) & (y_test == 0))
fn = np.sum((predictions == 0) & (y_test == 1))
tn = np.sum((predictions == 0) & (y_test == 0))

print(f"Accuracy:  {(tp + tn) / len(y_test):.1%}")
print(f"Precision: {tp / (tp + fp):.1%}  (of emails flagged as spam, how many were spam)")
print(f"Recall:    {tp / (tp + fn):.1%}  (of the spam emails, how many were caught)")
print()
print("Confusion matrix   predicted ham  predicted spam")
print(f"actual ham         {tn:>13}  {fp:>14}")
print(f"actual spam        {fn:>13}  {tp:>14}")

Accuracy:  88.8%
Precision: 91.3%  (of emails flagged as spam, how many were spam)
Recall:    79.4%  (of the spam emails, how many were caught)

Confusion matrix   predicted ham  predicted spam
actual ham                   286              15
actual spam                   41             158


## Comparison with baselines

To check the from-scratch implementation, it is compared with scikit-learn's `BernoulliNB` (the same model, so it should score the same), a logistic regression (a stronger linear model), and a classifier that always predicts the majority class.

In [4]:
from sklearn.naive_bayes import BernoulliNB
from sklearn.linear_model import LogisticRegression

results = {
    "Always predict ham (majority class)": np.mean(y_test == 0),
    "This classifier (from scratch)": np.mean(predictions == y_test),
    "scikit-learn BernoulliNB": BernoulliNB().fit(X_train, y_train).score(X_test, y_test),
    "scikit-learn LogisticRegression": LogisticRegression(max_iter=2000).fit(X_train, y_train).score(X_test, y_test),
}
for name, accuracy in results.items():
    print(f"{name:<38} {accuracy:.1%}")

Always predict ham (majority class)    60.2%
This classifier (from scratch)         88.8%
scikit-learn BernoulliNB               88.8%
scikit-learn LogisticRegression        92.6%


The from-scratch classifier matches scikit-learn's naive Bayes exactly. Logistic regression does better because it doesn't assume the features are independent: words like *free* and *money* tend to appear together, and naive Bayes counts that evidence twice.

## Fix made in October 2026

The original `predict` method (May 2024) looped over each email's features with `enumerate` and skipped index 0, treating it as the label column. But the label had already been removed before `predict` was called. As a result, the first feature was ignored and every other feature was scored with its neighbour's probabilities. Aligning the features with their probabilities raised test accuracy from **83.2% to 88.8%**. The method is also vectorised now, rather than looping in Python.